In [1]:
%pip install pandas
%pip install ollama


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip3.14 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip3.14 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('../aime_historical.csv')
display(df.tail(50))

,ID,Year,Problem Number,Question,Answer,Part
883,2022-II-9,2022,9,Let $\ell_A$ and $\ell_B$ be two distinct para...,244,II
884,2022-II-10,2022,10,Find the remainder when \[\binom{\binom{3}{2}}...,4,II
885,2022-II-11,2022,11,Let $ABCD$ be a convex quadrilateral with $AB=...,180,II
886,2022-II-12,2022,12,"Let $a, b, x,$ and $y$ be real numbers with $a...",23,II
887,2022-II-13,2022,13,There is a polynomial $P(x)$ with integer coef...,220,II
888,2022-II-14,2022,14,"For positive integers $a$ , $b$ , and $c$ with...",188,II
889,2022-II-15,2022,15,Two externally tangent circles $\omega_1$ and ...,140,II
890,2023-I-1,2023,1,Five men and nine women stand equally spaced a...,191,I
891,2023-I-2,2023,2,Positive real numbers $b \not= 1$ and $n$ sati...,881,I
892,2023-I-3,2023,3,"A plane contains $40$ lines, no $2$ of which a...",607,I


## Testing

Let's do a small test to see if our thinking traces works, we're calling the ollama API correctly, etc.

> NOTE: You will need to (a) have ollama installed, and (b) run `ollama run <model>` to grab the weights for inference.

In [4]:
%pip install transformers
%pip install torch
# restart the kernel after installing in your virtual environment to ensure the new packages are loaded


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip3.14 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip3.14 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "Qwen/Qwen3-4B"  # pull from HuggingFace
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, output_hidden_states=True)

inputs = tokenizer("Why is the sky blue?", return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

# outputs.hidden_states is a tuple: one tensor per layer
# each tensor shape: (batch, seq_len, hidden_dim)
hidden_states = outputs.hidden_states  

for layer_idx, layer_hs in enumerate(hidden_states):
    print(f"Layer {layer_idx}: {layer_hs.shape}")
    # e.g. Layer 0: torch.Size([1, 8, 2048])
    #               batch ^ ^ tokens ^ ^ ^ hidden_dim

# store as list of tensors per token across all layers:
# shape: [num_layers][batch, token, hidden_dim]
all_layers = [hs.squeeze(0) for hs in hidden_states]  # drop batch dim
# all_layers[layer][token] → hidden vector for that token at that layer

/home/avni/source/repos/cs639-final-project/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 3 files: 100%|██████████| 3/3 [06:42<00:00, 134.06s/it]
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 398/398 [00:01<00:00, 244.52it/s]
